# Task B -- context-conditional decode correction

A violent action word does not mean Violence; what it means depends on what else is in
the comment. Measured on the 420 training comments containing one:

| also contains | n | Others | Violence |
|---|---|---|---|
| media (BTV, channel, interview, Bigg Boss) | 78 | **0.27** | 0.18 |
| nothing | 216 | 0.06 | **0.34** |
| political | 71 | 0.00 | 0.15 (Political 0.56) |
| religion | 76 | 0.01 | 0.13 (Religion 0.61) |

So `hodibeku` next to BTV is Others; the same word with no target named is Violence.
`hastika.task_b.context_decode` learns a lift per cell and adds it to the model's log
probabilities after training. It changes no weights and costs no GPU.

**The open question.** On the calibrated TF-IDF SVM this was 0.5630 to 0.5823 nested,
with Violence F1 going 0.24 to ~0.35. Plain uncalibrated LinearSVC reached 0.5948, so
this is only a motivated decoder test, not evidence that the correction is already better.
But a transformer can represent
that interaction and a bag of n-grams cannot, so MuRIL may already know it. This notebook
finds out, on the real recipe.

`decode.py` already tried ONE weight per class and lost its nested check every time. This
differs because the correction flips sign with context, which a per-class weight cannot
express.

**Runtime** about 2.6 h: 45 min TAPT, 108 min for the five folds. Accelerator `GPU T4 x2`
or `GPU P100`, Internet on, Save Version -> Save & Run All.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. TAPT

Same corpus and flags as the submitted model. Skipped if the checkpoint already exists.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-ctxdecode"
TAPT_LOG = "artifacts/logs/ctxdecode_tapt.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/multiclass_train.csv", "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe", "--epochs", "8",
         "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "TAPT checkpoint was not written"
print("TAPT checkpoint ready:", TAPT_OUT)

## 2. Five folds of the current recipe

Five folds rather than a full fit, because the correction has to be fitted and nested on
out-of-fold probabilities and a full fit produces none. Deduplication is left ON here so
the OOF number is comparable with Run 6 and Run 7, which reported 0.6127 and 0.6082.
This is an OOF evaluation run, not the no-dedupe five-seed full-data submission; the
encoder recipe otherwise matches the 0.6410 submission.

In [ ]:
TAG = "b_ctx_5f"
TRAIN_LOG = pathlib.Path("artifacts/logs") / f"{TAG}.log"
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG, "--model", TAPT_OUT,
     "--folds", "5", "--reinit-layers", "1", "--rdrop", "0.5",
     "--aux-weight", "0", "--seeds", "42", "--epochs", "6", "--select", "last"],
    log=str(TRAIN_LOG))
RUN_DIR = pathlib.Path("artifacts/runs") / TAG
assert (RUN_DIR / "oof_probs.npy").exists(), "OOF probabilities were not written"
assert (RUN_DIR / "test_probs.npy").exists(), "validation probabilities were not written"
assert (RUN_DIR / "predictions.csv").exists(), "base predictions were not written"
print("five-fold outputs ready:", RUN_DIR)

## 3. Fit and nest the correction

`nested_score` picks the weight on an inner split of each training fold, so the rows being
scored never influence the weight chosen for them. Compare `corrected` against `plain`.

In [ ]:
import numpy as np, pandas as pd
from hastika.common.preprocessing import clean, dedupe_index
from hastika.task_b.context_decode import (cells, fit_lift, apply_lift, nested_score,
                                           select_weight, CONTEXTS)

LABELS = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
df = pd.read_csv("data/raw/multiclass_train.csv")
df = df.iloc[dedupe_index(df["Comment"].tolist(),
                          df["Hate Category"].tolist())].reset_index(drop=True)
X = [clean(t, demojize=True) for t in df["Comment"]]
y = df["Hate Category"].map(LABELS.index).values
oof = np.load(pathlib.Path("artifacts/runs") / TAG / "oof_probs.npy")
assert len(oof) == len(y), (oof.shape, len(y))

corrected, plain, chosen = nested_score(X, y, oof)
print(f"plain argmax      macro-F1 {plain:.4f}")
print(f"nested corrected  macro-F1 {corrected:.4f}   delta {corrected - plain:+.4f}")
print("weight chosen per fold:", chosen)

from sklearn.metrics import f1_score
c = cells(X)
lift = fit_lift(c, y, 6)
for w in [0.0, 0.1, 0.2, 0.3, 0.5, 0.8]:
    per = f1_score(y, apply_lift(oof, c, lift, w).argmax(1), average=None)
    print(f"  in-sample w={w:.1f}  macro {per.mean():.4f}   "
          + " ".join(f"{l[:5]}{v:.2f}" for l, v in zip(LABELS, per)))
print("\nin-sample numbers are optimistic; the nested one above is the result")

## 4. Write a submission only if the nested check wins

Same discipline as `decode.py`: if the correction does not beat plain argmax out of fold,
nothing is written. A rule that loses its nested check has never survived contact with
CodaBench in this project.

In [ ]:
ZIP = pathlib.Path("/kaggle/working/b_ctx_decoded.zip")
DECODED_WRITTEN = False
if corrected > plain:
    w, _ = select_weight(X, y, oof)
    test = np.load(pathlib.Path("artifacts/runs") / TAG / "test_probs.npy")
    ids = pd.read_csv("data/raw/multiclass_validation_inputs.csv")
    ct = cells([clean(t, demojize=True) for t in ids["Comment"]])
    pred = apply_lift(test, ct, lift, w).argmax(1)
    out_dir = pathlib.Path("artifacts/runs") / "b_ctx_decoded"
    out_dir.mkdir(parents=True, exist_ok=True)
    decoded = pd.DataFrame({"id": ids["id"],
                           "label": [LABELS[i] for i in pred]})
    assert len(decoded) == 395 and decoded["id"].is_unique
    assert set(decoded["label"]).issubset(LABELS)
    decoded.to_csv(out_dir / "predictions.csv", index=False)
    run([sys.executable, "-m", "hastika.common.submission", "--task", "b",
         "--pred", str(out_dir / "predictions.csv"), "--out", str(ZIP)])
    assert ZIP.exists(), "decoded submission ZIP was not written"
    with zipfile.ZipFile(ZIP) as z:
        assert z.namelist() == ["predictions.csv"], z.namelist()
    DECODED_WRITTEN = True
    base = pd.Series([LABELS[i] for i in test.argmax(1)])
    print(f"weight {w}; correction moved "
          f"{int((base.values != np.array([LABELS[i] for i in pred])).sum())} of 395 rows")
else:
    print(f"nested {corrected:.4f} does not beat plain {plain:.4f}; writing nothing.")
    print("Submit the existing b_reinit1_rdrop_full instead.")
    print("No decoded ZIP was created.")

## 5. Keep the outputs

`oof_probs.npy` is the reusable one: with it, any future decode idea can be tested on the
real model in seconds instead of 108 minutes.

In [ ]:
OUT = pathlib.Path("/kaggle/working/ctx_decode_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for p in [RUN_DIR / "oof_probs.npy", RUN_DIR / "test_probs.npy",
          RUN_DIR / "predictions.csv", TRAIN_LOG, pathlib.Path(TAPT_LOG)]:
    if p.exists():
        shutil.copy2(p, OUT / p.name)
if DECODED_WRITTEN and ZIP.exists():
    shutil.copy2(ZIP, OUT / ZIP.name)
print("download:", OUT)
print("files:", sorted(x.name for x in OUT.iterdir()))